# Mini Project: Sentiment Assistant with BERT Fine-Tuning

## Setup

In [ ]:
# Run once in a fresh environment
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## Imports & Hardware Check

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## Load the IMDB Reviews Dataset

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## Tokenizer Setup & Data Pipeline

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

## Initialize the Fine-Tuning Model

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

## Train and Monitor

In [ ]:
EPOCHS = 2  # increase to 3 if time allows

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("BERT Fine-Tuning on IMDB Reviews", fontsize=13)
plt.tight_layout()
plt.show()

## Evaluate on the Held-Out Test Set

In [ ]:
eval_metrics = model.evaluate(test_ds, return_dict=True)

print("Test set evaluation:")
for name, value in eval_metrics.items():
    print(f"  {name}: {value:.4f}")

If test accuracy crosses the ~0.90 benchmark, the model performs at a level comparable to standard classroom fine-tuning results on IMDB. For a real support team, this also means roughly 1 in 10 messages could still be misclassified, so a confidence threshold and human review step (discussed below) remain important before any fully automated escalation decision is made.

## Build a Reusable Inference Helper

In [ ]:
import numpy as np
import tensorflow as tf


def predict_sentiment(text: str):
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"
    )

    outputs = model(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
        token_type_ids=encoded["token_type_ids"]
    )

    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

    pred_idx = int(np.argmax(probs))
    label = "Positive" if pred_idx == 1 else "Negative"

    return label, float(probs.max())


custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

In [ ]:
# A few more support-style examples to stress-test the helper
support_examples = [
    "I've been waiting three weeks for a refund and nobody responds to my emails.",
    "Setup was quick and the dashboard looks great, really happy with it so far.",
    "The product works but the documentation is outdated and hard to follow."
]

for sentence in support_examples:
    label, confidence = predict_sentiment(sentence)
    print(f"'{sentence}'")
    print(f"  -> {label} (confidence={confidence:.3f})\n")

## Reflection & Next Steps

**What lever (data cleaning, hyperparameters, more epochs) most improved results?**

Using a pretrained BERT checkpoint was by far the biggest lever — it gave the model a strong understanding of language before seeing a single IMDB review, which is why only 2 epochs at a small learning rate (2e-5) were needed to reach high accuracy. Beyond that, the choice of `MAX_LENGTH` matters: too short truncates important context near the end of long reviews, while too long increases training time without much added benefit, since most of the sentiment signal in a review is usually expressed early or is repeated throughout.

**Where would you add guardrails before deploying this sentiment signal live?**

A confidence threshold should gate any fully automated action: messages where the model's top probability is only marginally above 0.5 (as seen with the orientation example above) should be routed to a human reviewer rather than auto-escalated or auto-closed. It would also be important to monitor for data drift — support messages differ from movie reviews in tone, length, and vocabulary (product names, error codes, account details), so performance should be validated on a labeled sample of real support transcripts before going live, not just assumed from IMDB accuracy.

**Which stakeholders benefit the most (support lead, product manager, compliance officer)?**

The support lead benefits most directly, since this signal lets them prioritize agent attention toward customers showing strong negative sentiment before they churn, rather than triaging tickets manually or by arrival order. The product manager benefits secondarily by being able to aggregate sentiment trends across feature releases or support topics to spot emerging pain points. A compliance officer would be more concerned with how the model is used (e.g., ensuring it doesn't unfairly deprioritize certain customer segments) and would want documentation of the model's accuracy, limitations, and the human-in-the-loop guardrails described above before approving production use.